In [ ]:
# Competition-Solution/notebooks/train/00_data_profiling/02_data_profiling_class_value_counts.ipynb

---

##### <b>Imports</b>

In [2]:

import sys
from rich.console import Console

sys.path.append("../../../src")
import data_utils

console = Console()

##### <b>Loading the data</b>

In [3]:
df_call2action, df_fdgo, df_violence = data_utils.load_competition_data(data_dir="../../../data/raw/", type="train")

Loading competition data...

Data of type 'train' loaded successfully.

##### <b>Class Value Counts</b>

In [4]:
print(f"Subtask 1 (Call2Action):")
print(df_call2action["C2A"].value_counts())
print(f"Subtask 2 (DBO):")
print(df_fdgo["DBO"].value_counts())
print(f"Subtask 3 (Violence):")
print(df_violence["VIO"].value_counts())

# Imbalance between the classes (value_counts_per_class / total_samples * 100)
c2a_imbalance = (df_call2action["C2A"].value_counts() / len(df_call2action)) * 100
print(c2a_imbalance)
dbo_imbalance = (df_fdgo["DBO"].value_counts() / len(df_fdgo)) * 100
print(dbo_imbalance)
vio_imbalance = (df_violence["VIO"].value_counts() / len(df_violence)) * 100
print(vio_imbalance)

Subtask 1 (Call2Action):
C2A
False    6177
True      663
Name: count, dtype: int64
Subtask 2 (DBO):
DBO
nothing       6277
criticism      804
agitation      313
subversive      60
Name: count, dtype: int64
Subtask 3 (Violence):
VIO
False    7219
True      564
Name: count, dtype: int64
C2A
False    90.307018
True      9.692982
Name: count, dtype: float64
DBO
nothing       84.209820
criticism     10.786155
agitation      4.199088
subversive     0.804937
Name: count, dtype: float64
VIO
False    92.753437
True      7.246563
Name: count, dtype: float64


<u><p>Interpretation of the class value counts</p></u>

*   **Subtask 1 (Call2Action - Label: `C2A`):**
    *   `False`: 6,177 instances *(~90.3%)*
    *   `True`: 663 instances *(~9.7%)*
    *   *Observation:* **Extreme class imbalance** consistent with the trial dataset proportions (trial: 90.3% vs 9.7%).

*   **Subtask 2 (DBO - Attacks on Democratic Basic Order - Label: `DBO`):**
    *   `nothing`: 6,277 instances *(~84.2%)*
    *   `criticism`: 804 instances *(~10.8%)*
    *   `agitation`: 313 instances *(~4.2%)*
    *   `subversive`: 60 instances *(~0.8%)*
    *   *Observation:* **Extreme class imbalance.** Less than the trial dataset but still high. The `subversive` class looks better in this regard (0.8% vs 0.4% in trial), but `agitation` remains severely underrepresented.

*   **Subtask 3 (Violence Detection - Label: `VIO`):**
    *   `False`: 7,219 instances *(~92.8%)*
    *   `True`: 564 instances *(~7.2%)*
    *   *Observation:* **High class imbalance** with slightly better positive class representation compared to the trial dataset (7.2% vs 5.7%).

<u><p>Summary & Implication</p></u>

*   *Summary of Observations:* All three subtasks in the training dataset maintain similar class imbalance patterns to the trial dataset, with the training set providing more examples of the minority classes in absolute terms.
*   *Implication:*
    *   **Macro-F1-Score Evaluation:** The consistent class imbalance patterns shows why the use of **Macro-F1 Score** as the primary evaluation metric is a good choice for the GermEval2025 competition, as standard accuracy would be misleading across all subtasks.

    *   **Enhanced Modeling Opportunities:** The larger absolute numbers of minority class examples (663 C2A positive, 60 DBO subversive, 564 VIO positive) provide better training opportunities compared to the trial dataset (102, 4, 60 respectively), while still unfortunately only making up a small fraction of the total data. This is likely going to make it difficult later on and might motivate data augmentation techniques.

    *   **Possible Methods that may make sense here:**
        *   **Stratified sampling** with larger sample sizes
        *   **Advanced resampling techniques** (SMOTE, ADASYN) with sufficient minority examples
        *   **Ensemble methods** combining multiple imbalance handling approaches
        *   **Cost-sensitive learning** with refined class weights (We ended up using this!)
        *   **Focal Loss** optimization with sufficient data for hyperparameter tuning (We ended up using this too!)

    *   **Subtask-Specific Considerations:**
        *   **DBO:** The `subversive` class, while still rare (0.8%), has 60 examples compared to 4 in trial dataset, making it potentially trainable with appropriate techniques. Nonetheless, this is likely to be the most challenging subtask to tackle. This is also reflected on the current leaderboard entries for the GermEval2025 competition, where DBO contributions consistently score a lower macro-F1 score than the other subtasks.
        *   **C2A & VIO:** The ~7-10% positive class ratios provide a reasonable foundation for binary classification with proper imbalance handling.

    *   **Multi-Task Learning Opportunity:** The imbalance patterns across subtasks, combined with the extensive overlap (62.74%), hint at multi-task learning approaches that can leverage shared representations while handling task-specific imbalances. If time allows it, we will also try to implement this.